# 01.04 - Optionality and Cardinality

> Builds on **01.03 — What Is Cardinality?** which introduces the concept of cardinality,
> why property graph databases do not enforce it, and what Orthograph uses it for.

orthograph distinguishes **three levels of optionality** when validating graph data:

1. **Property optionality** -- individual fields on a node or relationship can be required or optional.
2. **Entity-level optionality** -- an entire node or relationship *type* can be marked as required or optional in the model.
3. **Cardinality** -- constraints on how many relationships of a given type each node may have.

This notebook covers each level in isolation, then shows how they combine in practice.

In [ ]:
from typing import Optional

from orthograph.api.model import GraphValidator
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import (
    CardinalitySpec,
    NodeModel,
    RelationshipModel,
)

## Level 1: Property Optionality

Properties on `NodeModel` and `RelationshipModel` follow standard Pydantic rules:

- A field declared as `name: str` is **required** -- validation fails if it is missing.
- A field declared as `email: Optional[str] = None` is **optional** -- it can be absent or `None`.

This is the most granular level of optionality.

In [ ]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"

    name: str  # required
    age: int  # required
    email: Optional[str] = None  # optional


# Quick model to test node validation in isolation
prop_model = GraphDefinition(
    name="PropTest", node_types=[Person], relationship_types=[]
)
v = GraphValidator(prop_model)

# Valid: all required fields present, optional omitted
r = v.validate_nodes([{"__label__": "Person", "name": "Alice", "age": 32}])
print("All required present, optional omitted:", r.is_valid)

# Valid: optional field explicitly set to None
r = v.validate_nodes(
    [{"__label__": "Person", "name": "Alice", "age": 32, "email": None}]
)
print("Optional field set to None:            ", r.is_valid)

# Invalid: required field 'age' missing
r = v.validate_nodes([{"__label__": "Person", "name": "Alice"}])
print("Required field missing:                ", r.is_valid)
for err in r.errors:
    print(f"  -> {err.message}")

## Level 2: Entity-Level Optionality

Every `NodeModel` and `RelationshipModel` has a class variable `__optional__`
(default `True`). This controls whether the *type itself* must have at least one
instance present in the data.

- `__optional__ = True` (the default) -- the model defines what **can** exist.
  It is fine if the data contains zero instances of this type.
- `__optional__ = False` -- the model **requires** at least one instance of this
  type. Validation fails if the data is missing it entirely.

This is useful when you want to guarantee that certain entity types always appear,
for example to enforce that every valid graph contains at least one Person.

In [ ]:
class RequiredMovie(NodeModel):
    __label__ = "RequiredMovie"
    __uid_field__ = "title"
    __optional__ = False  # must have at least one instance

    title: str
    year: int


class OptionalCity(NodeModel):
    __label__ = "OptionalCity"
    __uid_field__ = "name"
    __optional__ = True  # this is the default, shown explicitly

    name: str
    country: str


entity_model = GraphDefinition(
    name="EntityTest",
    node_types=[RequiredMovie, OptionalCity],
    relationship_types=[],
)
v2 = GraphValidator(entity_model)

# Empty data: RequiredMovie is missing
r = v2.validate(nodes=[], relationships=[])
print("Empty data -- is_valid:", r.is_valid)
for err in r.errors:
    print(f"  [{err.code}] {err.message}")

print()

# Provide only RequiredMovie -- OptionalCity can be absent
r = v2.validate(
    nodes=[{"__label__": "RequiredMovie", "title": "Inception", "year": 2010}],
    relationships=[],
)
print("Only RequiredMovie present -- is_valid:", r.is_valid)

## Level 3: Cardinality

Cardinality specifies how many relationships of a given type a node can participate in.
It is expressed as a `CardinalitySpec(min, max)` where `max=None` means unbounded.

The four most common cardinalities are written directly as `CardinalitySpec` values:

| Notation | `CardinalitySpec` | Meaning |
|---|---|---|
| `0..1` | `CardinalitySpec(min=0, max=1)` | At most one |
| `1..1` | `CardinalitySpec(min=1, max=1)` | Exactly one |
| `0..*` | `CardinalitySpec(min=0, max=None)` | Any number (default) |
| `1..*` | `CardinalitySpec(min=1, max=None)` | At least one |

You can also create custom specs with `CardinalitySpec(min=..., max=...)`.

Cardinality is set per direction:
- `__source_cardinality__` constrains how many outgoing rels of this type each source node has.
- `__target_cardinality__` constrains how many incoming rels of this type each target node has.

In [ ]:
# Common cardinalities, written directly as CardinalitySpec values
common_specs = {
    "0..1": CardinalitySpec(min=0, max=1),
    "1..1": CardinalitySpec(min=1, max=1),
    "0..*": CardinalitySpec(min=0, max=None),
    "1..*": CardinalitySpec(min=1, max=None),
}
for notation, spec in common_specs.items():
    max_str = "N" if spec.max is None else str(spec.max)
    print(f"  {notation:6s}  CardinalitySpec  min={spec.min}  max={max_str}")

print()

# Custom cardinality
custom = CardinalitySpec(min=2, max=5)
print(f"Custom spec: {custom}")
print(f"  contains(1) = {custom.contains(1)}")
print(f"  contains(3) = {custom.contains(3)}")
print(f"  contains(6) = {custom.contains(6)}")

## Cardinality in Practice

Let's define a model where each Person must have exactly one LIVES_IN
relationship (`__source_cardinality__ = CardinalitySpec(min=1, max=1)`), meaning every Person
must live in exactly one City. We then validate data that violates this constraint
in both directions: too few and too many.

In [ ]:
class CPerson(NodeModel):
    __label__ = "CPerson"
    __uid_field__ = "name"
    name: str


class CCity(NodeModel):
    __label__ = "CCity"
    __uid_field__ = "name"
    name: str


class CLivesIn(RelationshipModel):
    __label__ = "C_LIVES_IN"
    __source_label__ = "CPerson"
    __target_label__ = "CCity"
    __source_cardinality__ = CardinalitySpec(
        min=1, max=1
    )  # each person -> exactly 1 city
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


card_model = GraphDefinition(
    name="CardinalityDemo",
    node_types=[CPerson, CCity],
    relationship_types=[CLivesIn],
)
v3 = GraphValidator(card_model)

base_nodes = [
    {"__label__": "CPerson", "name": "Alice"},
    {"__label__": "CCity", "name": "London"},
    {"__label__": "CCity", "name": "Paris"},
]

# --- Too few: Alice has 0 LIVES_IN (needs exactly 1) ---
r = v3.validate(nodes=base_nodes, relationships=[])
print("Too few (0 LIVES_IN):")
print(f"  is_valid: {r.is_valid}")
for err in r.errors:
    print(f"  [{err.code}] {err.message}")

print()

# --- Too many: Alice has 2 LIVES_IN (needs exactly 1) ---
r = v3.validate(
    nodes=base_nodes,
    relationships=[
        {
            "__label__": "C_LIVES_IN",
            "__source_uid__": "Alice",
            "__target_uid__": "London",
        },
        {
            "__label__": "C_LIVES_IN",
            "__source_uid__": "Alice",
            "__target_uid__": "Paris",
        },
    ],
)
print("Too many (2 LIVES_IN):")
print(f"  is_valid: {r.is_valid}")
for err in r.errors:
    print(f"  [{err.code}] {err.message}")

print()

# --- Just right: exactly 1 ---
r = v3.validate(
    nodes=base_nodes,
    relationships=[
        {
            "__label__": "C_LIVES_IN",
            "__source_uid__": "Alice",
            "__target_uid__": "London",
        },
    ],
)
print("Exactly 1 LIVES_IN:")
print(f"  is_valid: {r.is_valid}")

## Understanding `0..*`: Cardinality vs. Existence

A common question is: *"Why does `0..*` exist? If a node has zero
relationships of that type, doesn't that mean the relationship doesn't exist?"*

The answer lies in distinguishing two **orthogonal** concepts:

| Concept | What it controls | Mechanism |
|---|---|---|
| **Entity-level optionality** | Whether the relationship *type* must appear at all in the data | `__optional__ = True/False` |
| **Cardinality** | How many instances each *individual node* may have | `CardinalitySpec(min, max)` |

`CardinalitySpec(min=0, max=None)` (0..\*) means: *"this relationship type is defined in the schema,
but individual nodes are not required to participate in it."* A count of zero is a
valid state -- the node simply has no such relationship. This is **not** the same as
saying the relationship type doesn't exist.

`CardinalitySpec(min=1, max=None)` (1..\*) means: *"every node of this type must have at least one
instance of this relationship."* Zero would be a violation.

This is standard UML/ER notation -- `0..*` and `1..*` are both well-defined
multiplicities with distinct semantics.

The following example demonstrates the difference concretely.

In [ ]:
# --- Side-by-side: 0..* vs 1..* ---


class Employee(NodeModel):
    __label__ = "Employee"
    __uid_field__ = "name"
    name: str


class Project(NodeModel):
    __label__ = "Project"
    __uid_field__ = "name"
    name: str


# Relaxed: employees MAY work on projects (but don't have to)
class WorksOnRelaxed(RelationshipModel):
    __label__ = "WORKS_ON_R"
    __source_label__ = "Employee"
    __target_label__ = "Project"
    __source_cardinality__ = CardinalitySpec(min=0, max=None)  # 0..* -- optional
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


# Strict: every employee MUST work on at least one project
class WorksOnStrict(RelationshipModel):
    __label__ = "WORKS_ON_S"
    __source_label__ = "Employee"
    __target_label__ = "Project"
    __source_cardinality__ = CardinalitySpec(min=1, max=None)  # 1..* -- mandatory
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


relaxed_model = GraphDefinition(
    name="Relaxed",
    node_types=[Employee, Project],
    relationship_types=[WorksOnRelaxed],
)
strict_model = GraphDefinition(
    name="Strict",
    node_types=[Employee, Project],
    relationship_types=[WorksOnStrict],
)

nodes = [
    {"__label__": "Employee", "name": "Alice"},
    {"__label__": "Employee", "name": "Bob"},
    {"__label__": "Project", "name": "Alpha"},
]

# Only Alice works on Alpha; Bob has zero relationships
rels = [
    {"__label__": "WORKS_ON_R", "__source_uid__": "Alice", "__target_uid__": "Alpha"},
]
rels_strict = [
    {"__label__": "WORKS_ON_S", "__source_uid__": "Alice", "__target_uid__": "Alpha"},
]

# --- Relaxed model: Bob with 0 relationships is fine ---
r = GraphValidator(relaxed_model).validate(nodes, rels)
print("0..* (relaxed):")
print(f"  is_valid: {r.is_valid}")
print("  Bob has 0 WORKS_ON -- accepted (participation is optional)")

print()

# --- Strict model: Bob with 0 relationships is a violation ---
r = GraphValidator(strict_model).validate(nodes, rels_strict)
print("1..* (strict):")
print(f"  is_valid: {r.is_valid}")
for err in r.errors:
    print(f"  [{err.code}] {err.message}")
print("  Bob has 0 WORKS_ON -- rejected (participation is mandatory)")

### When to use each

**Use `0..*`** (the default) when:
- Validating **partial query results** where not all relationships are returned
- The relationship is genuinely optional (not every Person has a REVIEWED relationship)
- You want the schema to document what *can* exist without enforcing participation

**Use `1..*`** when:
- Every node of this type **must** participate (every Employee must WORK_AT somewhere)
- You are validating **canonical/complete data**, not query fragments
- The business rule requires mandatory participation with no upper bound

Both are valid, well-defined cardinalities. The choice depends on the strictness
of your domain rules and whether you expect complete or partial data.

## Combining Optionality Levels

In practice, these three levels work together. Consider a common scenario:
you receive partial data from a query and want a **relaxed** model that
accepts incomplete results.

- Some node types are **optional** (`__optional__ = True`) -- they might not appear.
- Some properties are **optional** (`Optional[T] = None`) -- they might be null.
- Cardinality is **relaxed** (`0..*`) -- we do not enforce relationship counts.

Contrast this with a **strict** model for canonical data, where types are required,
properties are mandatory, and cardinality is enforced.

In [ ]:
# -- A relaxed "query result" model --


class QRPerson(NodeModel):
    __label__ = "QRPerson"
    __uid_field__ = "name"
    __optional__ = True  # might not appear in partial results

    name: str
    age: Optional[int] = None  # age might not be projected
    email: Optional[str] = None


class QRMovie(NodeModel):
    __label__ = "QRMovie"
    __uid_field__ = "title"
    __optional__ = True

    title: str
    year: Optional[int] = None  # might not be projected
    rating: Optional[float] = None


class QRCity(NodeModel):
    __label__ = "QRCity"
    __uid_field__ = "name"
    __optional__ = True

    name: str
    country: Optional[str] = None


class QRActedIn(RelationshipModel):
    __label__ = "QR_ACTED_IN"
    __source_label__ = "QRPerson"
    __target_label__ = "QRMovie"
    __source_cardinality__ = CardinalitySpec(min=0, max=None)  # relaxed
    __target_cardinality__ = CardinalitySpec(min=0, max=None)

    role: Optional[str] = None  # role might not be projected


class QRLivesIn(RelationshipModel):
    __label__ = "QR_LIVES_IN"
    __source_label__ = "QRPerson"
    __target_label__ = "QRCity"
    __source_cardinality__ = CardinalitySpec(min=0, max=None)  # relaxed (not ONE)
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


query_model = GraphDefinition(
    name="QueryResult",
    node_types=[QRPerson, QRMovie, QRCity],
    relationship_types=[QRActedIn, QRLivesIn],
)
v_query = GraphValidator(query_model)

# Partial data: only persons and one relationship, no movies/cities present
partial_nodes = [
    {"__label__": "QRPerson", "name": "Alice"},  # age missing (optional)
    {"__label__": "QRPerson", "name": "Bob", "age": 28},
    {"__label__": "QRMovie", "title": "Inception"},  # year missing (optional)
]

partial_rels = [
    {
        "__label__": "QR_ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Inception",
    },
]

r = v_query.validate(partial_nodes, partial_rels)
print("Partial query result against relaxed model:")
print(f"  is_valid: {r.is_valid}")
print(f"  errors:   {len(r.errors)}")
print(f"  warnings: {len(r.warnings)}")
print()
print("All three optionality levels cooperate: optional types can be absent,")
print("optional properties can be null, and relaxed cardinality allows any count.")

## Level 4: Conditional Cardinality

Sometimes the allowed count for a relationship depends on **property values of
the endpoint nodes**, not just on the relationship type.  This is
*conditional cardinality* (ADR-029).

**Motivating example — `Operation.HAS_OUTPUT`:**

An `Operation` node has a `kind` property.  The number of allowed `HAS_OUTPUT`
edges depends jointly on the `kind` of the source `Operation` and the `kind` of
the target `Sample`:

| source `Operation.kind` | target `Sample.kind` | allowed outputs |
|---|---|---|
| `subsampling` | `subsampling` | 1..2 |
| `split` | `nothing` | 0..0 |
| `combine` | `nothing` | 0..0 |
| *(anything else)* | *(anything)* | 0..* (default) |

Express this as a `ConditionalCardinality` whose `rules` is a tuple of
`ConditionalRule(source=PropMatch({...}), target=PropMatch({...}), spec="...")`
entries, plus a required `default`.  Per ADR-032, `rule.source` always describes
the relationship's source-label node (`Operation`) and `rule.target` the
target-label node (`Sample`).
Every node-discriminator property used in the rules **must be required** on that
node type; the constructor rejects an optional discriminator at definition time.

In [ ]:
from orthograph.comparison.engine import compare_profile_to_definition
from orthograph.graph_definition.models import (
    CardinalitySpec,
    ConditionalCardinality,
    ConditionalRule,
    PropMatch,
)
from orthograph.graph_profile.models import (
    CardinalityStats,
    GraphProfile,
    NodeTypeProfile,
    RelationshipTypeProfile,
)


# --- Node types ---


class Operation(NodeModel):
    __label__ = "Operation"
    __uid_field__ = "uid"

    uid: str
    kind: str  # required discriminator


class Sample(NodeModel):
    __label__ = "Sample"
    __uid_field__ = "uid"

    uid: str
    kind: str  # required discriminator


# --- Relationship type with conditional cardinality ---


class HasOutput(RelationshipModel):
    __label__ = "HAS_OUTPUT"
    __source_label__ = "Operation"
    __target_label__ = "Sample"
    __source_cardinality__ = ConditionalCardinality(
        rules=(
            ConditionalRule(
                source=PropMatch({"kind": "subsampling"}),
                target=PropMatch({"kind": "subsampling"}),
                spec=CardinalitySpec(min=1, max=2),
            ),
            ConditionalRule(
                source=PropMatch({"kind": "split"}),
                target=PropMatch({"kind": "nothing"}),
                spec=CardinalitySpec(min=0, max=0),
            ),
            ConditionalRule(
                source=PropMatch({"kind": "combine"}),
                target=PropMatch({"kind": "nothing"}),
                spec=CardinalitySpec(min=0, max=0),
            ),
        ),
        default=CardinalitySpec(min=0, max=None),
    )


# --- Graph definition ---

op_model = GraphDefinition(
    name="OperationModel",
    node_types=[Operation, Sample],
    relationship_types=[HasOutput],
)
validator = GraphValidator(op_model)

print("Graph definition constructed successfully.")
print(f"HAS_OUTPUT source cardinality: {HasOutput.__source_cardinality__}")

In [ ]:
# --- Deciding scenario (from ADR-029): valid data ---
#
# Operation{subsampling} with 2 HAS_OUTPUT -> Sample{subsampling}  (bound: 1..2 -> OK)
# Operation{subsampling} with 1 HAS_OUTPUT -> Sample{nothing}      (bound: default 0..* -> OK)

nodes_valid = [
    {"__label__": "Operation", "uid": "op1", "kind": "subsampling"},
    {"__label__": "Sample", "uid": "s1", "kind": "subsampling"},
    {"__label__": "Sample", "uid": "s2", "kind": "subsampling"},
    {"__label__": "Sample", "uid": "s3", "kind": "nothing"},
]
rels_valid = [
    {"__label__": "HAS_OUTPUT", "__source_uid__": "op1", "__target_uid__": "s1"},
    {"__label__": "HAS_OUTPUT", "__source_uid__": "op1", "__target_uid__": "s2"},
    {"__label__": "HAS_OUTPUT", "__source_uid__": "op1", "__target_uid__": "s3"},
]

r = validator.validate(nodes=nodes_valid, relationships=rels_valid)
print("Deciding scenario — valid (2x subsampling + 1x nothing outputs):")
print(f"  is_valid: {r.is_valid}")
print(f"  errors:   {len(r.errors)}")

In [ ]:
# --- Violation scenario ---
#
# Operation{split} must have 0 HAS_OUTPUT -> Sample{nothing}, but has 1.

nodes_violation = [
    {"__label__": "Operation", "uid": "op2", "kind": "split"},
    {"__label__": "Sample", "uid": "s4", "kind": "nothing"},
]
rels_violation = [
    {"__label__": "HAS_OUTPUT", "__source_uid__": "op2", "__target_uid__": "s4"},
]

r = validator.validate(nodes=nodes_violation, relationships=rels_violation)
print("Violation scenario — split op has 1 nothing-output (must be 0):")
print(f"  is_valid: {r.is_valid}")
for issue in r.errors:
    print(f"  [{issue.code}] {issue.message}")

In [ ]:
# --- UNVERIFIABLE finding when comparing against a live-DB profile ---
#
# The in-memory validator enforces conditional cardinality per node.
# A live-DB profile only has aggregate cardinality stats (min/max degree
# across all pairs), not per-pair counts.  When the declared side is
# CARDINALITY_UNVERIFIABLE (INFO) instead of attempting a false comparison.

profile = GraphProfile(
    source="synthetic",
    node_type_profiles={
        "Operation": NodeTypeProfile(label="Operation", count=3),
        "Sample": NodeTypeProfile(label="Sample", count=5),
    },
    rel_type_profiles={
        "HAS_OUTPUT": RelationshipTypeProfile(
            rel_type="HAS_OUTPUT",
            count=6,
            source_labels={"Operation"},
            target_labels={"Sample"},
            cardinality_stats=CardinalityStats(count=3, min=0, max=3, mean=2.0),
        )
    },
)

comparison = compare_profile_to_definition(profile, op_model)
print("Comparison result (conditional side vs. aggregate profile):")
print(f"  is_valid: {comparison.is_valid}")
unverifiable = [i for i in comparison.issues if i.code == "CARDINALITY_UNVERIFIABLE"]
print(f"  CARDINALITY_UNVERIFIABLE issues: {len(unverifiable)}")
for issue in unverifiable:
    print(f"  [{issue.severity.value}] {issue.message}")
print()
print(
    "Per-pair enforcement against a live database is delivered by E41 (ADR-030),"
    "\nwhich adds partitioned_cardinality stats to the profile."
)